In [ ]:
print("Mushroom Classifier")

In [12]:
#common imports and variables

import pickle
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import pickle
import time

training_times = {}
labels = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1-Score",
    "AUC Score",
    "Matthews Corr (MCC)"
]

def model_performance_analysis(y_test,y_pred,y_probs):
    accuracy=accuracy_score(y_test, y_pred)
    precision=precision_score(y_test, y_pred)
    recall=recall_score(y_test, y_pred)
    f1=f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_probs)
    mcc=matthews_corrcoef(y_test, y_pred)

    score_matrix = [accuracy, precision, recall, f1, auc, mcc]
    df_scores = pd.DataFrame({"Score": score_matrix}, index=labels)
    print(df_scores.round(5))

    cm = confusion_matrix(y_test, y_pred)
    class_labels = ["Edible", "Poisonous"]
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_labels,
        yticklabels=class_labels,
    )

    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title("Confusion Matrix")
    plt.show()



In [2]:
# https://www.kaggle.com/datasets/uciml/mushroom-classification
# Preparing data

from sklearn.model_selection import train_test_split
df = pd.read_csv("mushrooms.csv")

train_data, test_data = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["class"]
)
test_data.to_csv("test_data.csv", index=False)



In [3]:
## Encode and Scale data for the Logistic Regression , K-NN Algos and Naive Baye's

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder,OneHotEncoder

# Seperate the Input features and Output class of the Training data
X_train = train_data.drop("class", axis=1)
y_train = train_data["class"]

# Seperate the Input features and Output class of the Test data
X_test = test_data.drop("class", axis=1)
y_test = test_data["class"]

#Define the Encoders and the Scalers
oneHotEncoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)
label_encoder = LabelEncoder()
scaler = StandardScaler()

#Encode the Input features of the Training data using OneHotEncoder
X_train_ohe = oneHotEncoder.fit_transform(X_train)
#Encode the Output class of the Training data using LabelEncoder
y_train_le = label_encoder.fit_transform(y_train)
#Scale the Input features of the Training data
X_train_ohe_scaled = scaler.fit_transform(X_train_ohe)


# Save the fitted encoder,label_encoder & scaler to a file for later prediction
with open('./trained_models/encoder.pkl', 'wb') as f:
    pickle.dump(oneHotEncoder, f)
with open('./trained_models/label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
with open('./trained_models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)


#Encode the Input features of the Test data using OneHotEncoder
X_test_ohe = oneHotEncoder.transform(X_test)
#Scale the Input features of the Training data
X_test_ohe_scaled = scaler.transform(X_test_ohe)
#Encode the Output class of the Test data using LabelEncoder
y_test_le = label_encoder.transform(y_test)



In [ ]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=20)
start = time.time()
lr.fit(X_train_ohe_scaled, y_train_le)
training_times['Logistic Regression'] = time.time() - start

y_pred_lr = lr.predict(X_test_ohe_scaled)

y_probs_lr = lr.predict_proba(X_test_ohe_scaled)[:, 1]
model_performance_analysis(y_test_le, y_pred_lr,y_probs_lr)

# Save the trained model as a pickle file
with open("./trained_models/logistic_regression_model.pkl", "wb") as file:
    pickle.dump(lr, file)

In [ ]:
# kNN
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
start = time.time()
knn.fit(X_train_ohe_scaled, y_train_le)
training_times['K-NN'] = time.time() - start

y_pred_knn = knn.predict(X_test_ohe_scaled)

y_probs_knn = knn.predict_proba(X_test_ohe_scaled)[:, 1]
model_performance_analysis(y_test_le, y_pred_knn,y_probs_knn)

# Save the trained model as a pickle file
with open("./trained_models/knn_model.pkl", "wb") as file:
    pickle.dump(knn, file)

In [6]:
## Encode data for the Tree based algos (Decision Tree and Random Forest)
# Convert every Input feature from text strings to numeric categories instantly for Training Data
X_train_oe = train_data.drop('class', axis=1).apply(lambda x: x.astype('category').cat.codes)

# Convert every every Input feature from text strings to numeric categories instantly for Test Data
X_test_oe = test_data.drop('class', axis=1).apply(lambda x: x.astype('category').cat.codes)


In [ ]:
# Decision Tree
from sklearn.tree import DecisionTreeClassifier, export_text

dt = DecisionTreeClassifier(random_state=42)
start = time.time()
dt.fit(X_train_oe, y_train_le)
training_times['Decision Tree'] = time.time() - start

y_pred_dt = dt.predict(X_test_oe)

y_probs_dt = dt.predict_proba(X_test_oe)[:, 1]
model_performance_analysis(y_test_le, y_pred_dt,y_probs_dt)

# Save the trained model as a pickle file
with open("./trained_models/decision_tree_model.pkl", "wb") as file:
    pickle.dump(dt, file)



#tree_rules = export_text(dt,feature_names=list(X.columns))
#print(tree_rules)

In [ ]:
#Naive Bayes Classifier - Multinomial
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()
start = time.time()
nb.fit(X_train_ohe, y_train_le)
training_times['Naive Bayes'] = time.time() - start

y_pred_nb = nb.predict(X_test_ohe)

y_probs_nb = nb.predict_proba(X_test_ohe)[:, 1]
model_performance_analysis(y_test_le, y_pred_nb,y_probs_nb)

# Save the trained model as a pickle file
with open("./trained_models/naive_bayes_gaussian_model.pkl", "wb") as file:
    pickle.dump(nb, file)

In [ ]:
from sklearn.ensemble import RandomForestClassifier


rf = RandomForestClassifier(n_estimators=100,random_state=42)
start = time.time()
rf.fit(X_train_oe, y_train_le)
training_times['Random Forest'] = time.time() - start

y_pred_rf = rf.predict(X_test_oe)

y_probs_rf = rf.predict_proba(X_test_oe)[:, 1]
model_performance_analysis(y_test_le, y_pred_rf,y_probs_rf)

# Save the trained model as a pickle file
with open("./trained_models/random_forest_model.pkl", "wb") as file:
    pickle.dump(rf, file)

In [ ]:
print("Training Time Comparison:")
print("=" * 30)
for model, duration in training_times.items():
    print(f"{model:20}: {duration:.6f} seconds")

In [ ]:
#Overfitting Check: Training vs. Test Performance
results = []

# Helper to calculate and store training vs test accuracy
def check_overfitting(name, model, X_train, y_train, X_test, y_test):
    train_acc = model.score(X_train, y_train)
    test_acc = model.score(X_test, y_test)
    results.append({
        'Model': name,
        'Train Accuracy': round(train_acc, 5),
        'Test Accuracy': round(test_acc, 5),
        'Gap': round(train_acc - test_acc, 5)
    })

# Check all models
check_overfitting('Logistic Regression', lr, X_train_ohe_scaled, y_train_le, X_test_ohe_scaled, y_test_le)
check_overfitting('K-NN', knn, X_train_ohe_scaled, y_train_le, X_test_ohe_scaled, y_test_le)
check_overfitting('Decision Tree', dt, X_train_oe, y_train_le, X_test_oe, y_test_le)
check_overfitting('Naive Bayes', nb, X_train_ohe, y_train_le, X_test_ohe, y_test_le)
check_overfitting('Random Forest', rf, X_train_oe, y_train_le, X_test_oe, y_test_le)

# Display comparison table
overfit_df = pd.DataFrame(results)
display(overfit_df)